<a href="https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")






Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

 **Classification:**  to group  pages into "high position" vs "low position" then see which signals predict the bucket. Cleaner metric, easier to defend, easier to turn into an action 2.didnt understnad what is target or proxy

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

# swap 'content_id' / 'page_id' for whatever the actual ID column is called
print("Total rows:", len(df))
print("Unique pages:", df['content_id'].nunique())

Rows: 30000, Columns: 44
Total rows: 30000
Unique pages: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy = since we cannot directly measure "is this a good page," we use a stand-in number that's close enough
My target is a proxy I define: whether a page is in the top 10 search positions. It's built from the observed position column, which comes directly from Google I'm not predicting something invented, I'm just drawing a line through a real number to make it something a classifier can learn.

In [ ]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

# swap 'content_id' / 'page_id' for whatever the actual ID column is called
print("Total rows:", len(df))
print("Unique pages:", df['content_id'].nunique())

print(df.columns.tolist())
df['is_top10'] = (df['avg_position'] <= 10).astype(int)
df['is_top10'].value_counts(normalize=True)

Rows: 30000, Columns: 44
Total rows: 30000
Unique pages: 30000
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,proportion
is_top10,
0,0.527067
1,0.472933


* content_id is the unique key — good, matches what you already confirmed (30,000 rows = 30,000 unique pages).



You have real on-page signals: word_count, char_count, content_age_days, days_since_last_update — these are the "on-page" side you mentioned in section
*   You don't have classic off-page signals like backlinks or domain authority in this columns list — so if your section 1 markdown mentioned "off-page" signals, you may want to soften that to just "on-page and behavioral signals" (things like ctr, engagement_rate, ai_traffic_pct) since that's what's actually here. Small honesty fix, but the assignment explicitly asks for "honest numbers.

## 3. Success metric

*One metric you can defend. What number means 'good'?*


1. Checking of the  class balance



*One metric you can defend. What number means 'good'?*

1. Class balance: 52.7% of pages are NOT top-10, 47.3% are top-10 — a nearly even
split. Majority-class baseline accuracy = 0.527.

2. Metric: accuracy, paired with F1 on the top-10 class so I can see the model is
actually finding top-10 pages and not just leaning on the balanced split.

3. What "good" means: beating the 0.527 baseline by a clear margin — I'd consider
accuracy above ~0.65-0.70 a meaningful result, since it would show the signals carry
real information about position rather than noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline = df['is_top10'].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {baseline:.3f}")

Majority-class baseline accuracy: 0.527


10-20% of pages are top-10, and 80-90% are not. That imbalance matters — it means accuracy is a trap. A model that just guesses "not top-10" every time could hit 85% accuracy and be useless.

So propose this instead: recall on the top-10 class — of all the pages that are actually top-10, what fraction does your model correctly flag? Pair it with precision (of the pages it flags as top-10, how many really are) so it's not just guessing "yes" for everything either. If you want a single number, use F1 score on the top-10 class.

Your "good" number is: beats the baseline of predicting the majority class every time (that baseline recall/precision on the minority class is 0, so any real lift is progress) — say something like "F1 above 0.3-0.4 would mean it's finding real signal, not noise.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

My unit of analysis is one page (one URL). Each row in this dataset represents a single piece of content and its on-page/off-page signals plus its search position at the time of the snapshot."

(Adjust that if your CSV actually has one row per page-per-date instead of just one row per page — check by seeing if the same URL/page ID repeats across rows.)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rows = len(df)
unique_pages = df['content_id'].nunique()
print(f"Rows: {rows}, Unique pages: {unique_pages}")
print("One row = one page:", rows == unique_pages)

Rows: 30000, Unique pages: 30000
One row = one page: True


If those two numbers match (rows = unique pages), that confirms "one row = one page" — put that confirmation as a comment or print statement, since it's your evidence, not just an assumption.

Run it and tell me:

What the actual column names are (paste df.columns.tolist() output)
Whether rows == unique page count

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



*What makes the pattern too messy for an if-statement?*

The correlations between individual signals and avg_position are all weak (roughly -0.15 to 0.2, none above ~0.4-0.5) — no single column crosses a threshold that would justify a rule like "if word_count > X, top-10." The real pattern is probably in combinations of signals (e.g. long content + many internal links + high engagement), which a fixed if-statement can't express but a model can learn.
\

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols].corr()['avg_position'].sort_values()

,avg_position
is_top10,-0.663994
days_with_sessions,-0.106046
clicks_90d,-0.099304
clicks_last_30d,-0.092970
clicks_prev_30d,-0.091237
ctr,-0.072590
impressions_90d,-0.070786
impressions_prev_30d,-0.069516
impressions_last_30d,-0.067642
sessions_prev_30d,-0.057640


Look at the resulting numbers. If you see correlations like 0.1, -0.15, 0.2 (weak, none above ~0.4-0.5), that's your evidence: no single signal is strong enough to write a rule like "if word_count > X, top-10." The real pattern likely lives in combinations of signals (e.g. long content + many internal links + short title), which is exactly what a rule can't express but a model (like a decision tree or random forest) can learn automatically

Excluding is_top10 (which is derived from avg_position itself), every individual signal correlates weakly with avg_position — the strongest is content_age_days at 0.158, followed by age_tier_order (0.152) and word_count (0.124); everything else is below 0.11. No single feature comes close to a threshold like "if word_count > X, top-10" — a rule that simple would miss almost all the signal. The real pattern likely lives in combinations of weak signals (e.g. older content + longer word count + more days since update), which is exactly what a fixed if-statement can't express but a model can learn.

## Self-check

Before you submit, confirm each line honestly:

- [ *] Every section above is filled — markdown thinking AND the code that backs it
- [ *] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ *] No client names, URLs, or private queries anywhere
- [ *] My claims use careful words: observed, measured, directional, decision-support
- [* ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.